In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")  
books = pd.read_csv(DATA_DIR / "books_with_categories.csv")  

In [2]:
from transformers import pipeline
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = "cpu")
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528684265911579},
  {'label': 'neutral', 'score': 0.005764580797404051},
  {'label': 'anger', 'score': 0.004419783595949411},
  {'label': 'sadness', 'score': 0.002092391485348344},
  {'label': 'disgust', 'score': 0.001611991785466671},
  {'label': 'fear', 'score': 0.0004138525982853025}]]

In [3]:
import re

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]


def split_sentences(text):
    """Split after . ! ? and never return empty strings."""  # CHANGED: was text.split(".")
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]


def calculate_max_emotion_scores(predictions):
    """Max score per emotion across sentences, matched BY LABEL NAME."""
    max_scores = {label: 0.0 for label in emotion_labels}
    for prediction in predictions:  # CHANGED: no more sorted() + positional index
        for item in prediction:
            label = item["label"]
            max_scores[label] = max(max_scores[label], item["score"])
    return max_scores

In [4]:
from tqdm.notebook import tqdm

isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = split_sentences(books["description"][i])  
    predictions = classifier(sentences, truncation=True) 
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

  0%|          | 0/5197 [00:00<?, ?it/s]

c:\Users\lenovo\OneDrive\Desktop\Book Recommender\myenv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  329MB            

model.safetensors: downloading bytes:           |  0.00B            

c:\Users\lenovo\OneDrive\Desktop\Book Recommender\myenv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--j-hartmann--emotion-english-distilroberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [5]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.016755,0.454295,0.979017,0.860424,0.939770,0.721434,0.730330,9780002005883
1,0.615502,0.711034,0.922018,0.661608,0.037852,0.168005,0.891110,9780002261982
2,0.035669,0.042248,0.970825,0.760198,0.011307,0.009915,0.056921,9780006178736
3,0.189178,0.234425,0.407684,0.209943,0.051571,0.018530,0.796719,9780006280897
4,0.041969,0.209280,0.064400,0.120436,0.406045,0.016177,0.823639,9780006280934
...,...,...,...,...,...,...,...,...
5192,0.302140,0.133402,0.894961,0.303278,0.968084,0.020487,0.837031,9788172235222
5193,0.042661,0.162955,0.019944,0.398245,0.014484,0.202333,0.899161,9788173031014
5194,0.011246,0.019147,0.236123,0.942169,0.060888,0.050320,0.479330,9788179921623
5195,0.028676,0.137220,0.332192,0.616375,0.250337,0.047061,0.953387,9788185300535


In [6]:
books = pd.merge(books, emotions_df, on="isbn13")
assert len(books) == len(emotions_df) 
books.to_csv(DATA_DIR / "books_with_emotions.csv", index=False)  